# NoSQL & 缓存

> **适用场景**: 高并发读取、低延迟访问、特殊数据结构
> **面试频率**: ⭐⭐⭐⭐ 高频

## 目录
1. CAP / PACELC 定理
2. Redis：数据结构 & 持久化 RDB/AOF
3. Cassandra：Partition Key 设计
4. DynamoDB：GSI / LSI / Capacity Modes
5. 练习题

---
## 1. CAP / PACELC 定理

### CAP 定理
分布式系统在网络分区（P）发生时，只能在以下两者选一：

| 属性 | 含义 |
|------|------|
| **C**onsistency | 所有节点看到同样的数据（强一致性）|
| **A**vailability | 每个请求都有响应（不一定最新）|
| **P**artition tolerance | 网络分区时系统仍能运行 |

**实际选择**：P 是必须的（网络分区无法避免），所以真正的选择是 **CP vs AP**。

```
CP 系统：HBase, Zookeeper, Spanner
  → 网络分区时拒绝写入，保证已有数据一致
  → 适合：金融交易、库存系统

AP 系统：Cassandra, DynamoDB, CouchDB
  → 网络分区时仍可写入，但可能读到旧数据
  → 适合：用户行为日志、购物车、社交 Feed
```

### PACELC 定理（CAP 的扩展）
即使没有网络分区，也面临 Latency vs Consistency 的权衡：
- **PA/EL**：分区时选 A，正常时选低延迟（如 Cassandra 默认）
- **PC/EC**：分区时选 C，正常时选一致性（如 Spanner、Zookeeper）

数据工程实践：大多数数据湖/数仓场景选 **AP + 最终一致性**，业务关键路径选 **CP**。

---
## 2. Redis：数据结构 & 持久化 RDB/AOF

### 核心数据结构

| 类型 | 命令示例 | 数据工程使用场景 |
|------|----------|------------------|
| **String** | `SET key val EX 3600` | 缓存 API 结果、计数器 |
| **Hash** | `HSET user:1 name Alice age 30` | 缓存对象（如用户信息）|
| **List** | `LPUSH queue job1; RPOP queue` | 消息队列（简单场景）|
| **Set** | `SADD online_users uid1 uid2` | 去重、UV 统计 |
| **Sorted Set** | `ZADD leaderboard 100 alice` | 排行榜、按时间排序的事件 |
| **HyperLogLog** | `PFADD hll:page uid1 uid2` | 大规模基数估算（UV，误差 < 1%）|
| **Stream** | `XADD mystream * field val` | 持久化消息队列（Kafka 轻量替代）|

```python
import redis
r = redis.Redis(host='localhost', port=6379, decode_responses=True)

# 缓存 DB 查询结果
cache_key = f'metrics:daily:{date}'
cached = r.get(cache_key)
if not cached:
    result = db.query('SELECT ...')
    r.setex(cache_key, 3600, json.dumps(result))  # TTL = 1小时
    return result
return json.loads(cached)
```

### 持久化：RDB vs AOF

| 方式 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **RDB** | 定时 fork 进程，生成内存快照 | 恢复快，文件小 | 可能丢失最近数据（快照间隔内）|
| **AOF** | 记录每条写命令（追加日志）| 数据安全（可配置 fsync 频率）| 文件大，恢复慢 |
| **RDB + AOF** | 两者结合 | 兼顾恢复速度和数据安全 | 推荐生产配置 |

```conf
# redis.conf
# RDB 配置
save 900 1      # 900秒内至少1次写 → 触发快照
save 300 10     # 300秒内至少10次写
save 60 10000   # 60秒内至少10000次写

# AOF 配置
appendonly yes
appendfsync everysec  # 每秒 fsync（平衡性能和安全）
# appendfsync always  # 每条命令 fsync（最安全，最慢）
# appendfsync no      # 由 OS 决定（最快，可能丢数据）
```

---
## 3. Cassandra：Partition Key 设计

### 数据模型基础
Cassandra 的核心是 **Query-driven Design**：先想清楚查询模式，再设计表结构（与 RDBMS 相反）。

**Primary Key 构成**：
```cql
PRIMARY KEY ((partition_key), clustering_key1, clustering_key2)
-- partition_key：决定数据存在哪个节点（哈希分区）
-- clustering_key：决定分区内数据的排序
```

### Partition Key 设计原则

**原则 1：高基数，分布均匀**
```cql
-- ❌ 差：按 country 分区（国家只有几百个，严重倾斜）
PRIMARY KEY (country, user_id, timestamp)

-- ✅ 好：按 user_id 分区（百万级，均匀分布）
PRIMARY KEY (user_id, timestamp)
```

**原则 2：避免超大分区（Hot Partition）**
```cql
-- ❌ 问题：所有事件按天分区，高流量时单天数据量巨大
PRIMARY KEY (date, event_id)

-- ✅ 解决：加 bucket 限制分区大小
PRIMARY KEY ((date, bucket), event_id)
-- bucket = event_id % 10  -- 将每天数据分散到10个分区
```

**原则 3：一个查询一张表**
```cql
-- 场景：需要按 user_id 查询，也需要按 email 查询
-- 创建两张表！

-- 按 user_id 查询
CREATE TABLE users_by_id (
    user_id UUID PRIMARY KEY,
    email TEXT,
    name TEXT
);

-- 按 email 查询
CREATE TABLE users_by_email (
    email TEXT PRIMARY KEY,
    user_id UUID,
    name TEXT
);
```

### 一致性级别
```cql
-- Cassandra 可以按操作级别设置一致性
CONSISTENCY QUORUM;   -- 多数节点响应（推荐，平衡性能和一致性）
CONSISTENCY ONE;      -- 单节点响应（最快）
CONSISTENCY ALL;      -- 全部节点（最强一致性，性能差）

-- 写入 QUORUM + 读取 QUORUM = 强一致性（W + R > N）
```

---
## 4. DynamoDB：GSI / LSI / Capacity Modes

### 基本结构
```
Table: orders
├── Primary Key
│   ├── Partition Key: customer_id
│   └── Sort Key: order_date
├── LSI（本地二级索引）
└── GSI（全局二级索引）
```

### LSI vs GSI

| 特性 | LSI (Local Secondary Index) | GSI (Global Secondary Index) |
|------|----------------------------|------------------------------|
| Partition Key | **必须与主表相同** | 可以完全不同 |
| Sort Key | 可以不同 | 可以不同 |
| 创建时机 | **只能在建表时创建** | 任何时候都可以创建 |
| 一致性 | 支持强一致性读 | 只支持最终一致性读 |
| 存储 | 与主表共享 10GB 分区限制 | 独立存储，无大小限制 |
| 适用场景 | 同一 PK 下不同排序 | 完全不同的查询维度 |

```python
# GSI 示例：orders 表同时支持按 customer_id 和按 product_id 查询
table = dynamodb.create_table(
    TableName='orders',
    KeySchema=[
        {'AttributeName': 'customer_id', 'KeyType': 'HASH'},
        {'AttributeName': 'order_date', 'KeyType': 'RANGE'}
    ],
    GlobalSecondaryIndexes=[
        {
            'IndexName': 'product-index',
            'KeySchema': [
                {'AttributeName': 'product_id', 'KeyType': 'HASH'},
                {'AttributeName': 'order_date', 'KeyType': 'RANGE'}
            ],
            'Projection': {'ProjectionType': 'ALL'}
        }
    ]
)
```

### Capacity Modes

**Provisioned Throughput（预置容量）**
- 手动设置 RCU（读容量单元）和 WCU（写容量单元）
- 超出时限流（ThrottlingException）
- 适合：流量可预测、稳定的工作负载
- 可配合 Auto Scaling 自动调整

**On-Demand（按需模式）**
- 无需预置，按实际请求计费
- 自动扩缩容，无限流
- 适合：流量波动大、新产品、不可预测的负载
- 费用通常更高（约 7x 单位价格）

```python
# 查询 GSI
response = table.query(
    IndexName='product-index',
    KeyConditionExpression=Key('product_id').eq('P001') &
                           Key('order_date').between('2024-01-01', '2024-12-31')
)
```

---
## 5. 练习题

### Q1 CAP 定理中为什么说 CP vs AP，而不是 CA vs P？

<details><summary>参考答案</summary>

因为在分布式系统中，**网络分区（P）是客观存在的**，不可避免。即使是内网也会有网络抖动、节点故障。

CA（放弃 P）意味着系统在网络分区时完全不可用，这实际上就是单节点数据库（如单机 MySQL），不是真正的分布式系统。

所以实际上是：当 P 发生时，在 C（拒绝服务保一致）和 A（继续服务但可能读到旧数据）之间选择。
</details>

---

### Q2 [高频] Redis 作为缓存使用时如何防止缓存穿透、缓存击穿、缓存雪崩？

<details><summary>参考答案</summary>

**缓存穿透**（查不存在的 key，每次都打到 DB）：
- 缓存空值（`SET key '' EX 60`）
- 布隆过滤器（Bloom Filter）提前过滤不存在的 key

**缓存击穿**（热点 key 过期，大量请求同时打 DB）：
- 互斥锁（setnx 加锁，只有一个请求回源 DB）
- 热点 key 永不过期（异步更新）

**缓存雪崩**（大量 key 同时过期，DB 被压垮）：
- TTL 加随机抖动（`EX 3600 + random(0, 600)`）
- Redis 集群（高可用，避免单点故障）
- 降级：Redis 故障时返回默认值，不透传到 DB
</details>

---

### Q3 Cassandra 为什么不推荐 ALLOW FILTERING？

<details><summary>参考答案</summary>

`ALLOW FILTERING` 会强制 Cassandra 对**所有节点**进行全表扫描，然后在内存中过滤。因为 Cassandra 不支持非分区键的过滤（没有索引时），`ALLOW FILTERING` 实际上是全集群扫描，数据量大时性能极差（O(n) 扫描，且会给所有节点带来压力）。

正确做法：为查询需求创建新表或 Materialized View，查询模式驱动数据模型设计。
</details>

---

### Q4 [高频] DynamoDB GSI 和 LSI 如何选择？

<details><summary>参考答案</summary>

- **选 LSI**：需要同一 PK 下按不同属性排序查询，且建表时就能确定，需要强一致性读
- **选 GSI**：需要完全不同的查询维度（不同 PK），或者表已经存在需要后加索引

实践中 **GSI 更常用**（灵活、可随时创建）。LSI 的主要限制是 10GB 分区大小上限（与主表共享），生产环境容易成为瓶颈。

GSI 注意事项：
- GSI 的 PK 可以重复（与主表不同）
- GSI 只支持最终一致性读（不支持强一致性）
- GSI 单独计费 WCU/RCU
</details>

---

### Q5 在数据工程中，Redis 和 Kafka 都能做消息队列，如何选择？

<details><summary>参考答案</summary>

| 维度 | Redis Stream/List | Kafka |
|------|-------------------|-------|
| 消息持久化 | 内存为主（RDB/AOF 可持久化）| 磁盘持久化，可保留数天/周 |
| 消费者组 | 支持（Stream 功能）| 原生支持，功能完整 |
| 吞吐量 | 中等（单机百万 QPS）| 极高（集群可达千万）|
| 消息回溯 | Stream 支持，但受内存限制 | 任意时间回溯 |
| 运维复杂度 | 低 | 高（需要 ZooKeeper/KRaft）|

选 Redis：简单场景，消息量不大，不需要长时间保留，已有 Redis 基础设施

选 Kafka：大规模数据管道，需要消息回溯，多消费者，高可靠性要求
</details>